#  Notebook 03 — Exploratory Data Analysis (EDA)
## Spotify Music Analytics Dataset (2015–2025)

**Goal:** Visualise the cleaned dataset from every angle — distributions, correlations, categorical breakdowns, and time trends — to find patterns that will drive the Streamlit dashboard.

> 15 charts, 5 analytical sections. Every chart is followed by a short interpretation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Global style settings
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
PALETTE = sns.color_palette('Set2')
SPOTIFY_GREEN = '#1DB954'

# Load cleaned data
df = pd.read_csv('../data/processed/spotify_cleaned.csv', parse_dates=['release_date'])
df['popularity_tier'] = pd.Categorical(df['popularity_tier'],
                                        categories=['Low','Medium','High'], ordered=True)
df['energy_level'] = pd.Categorical(df['energy_level'],
                                     categories=['Low','Medium','High'], ordered=True)

AUDIO_FEATURES = ['danceability', 'energy', 'loudness', 'instrumentalness', 'tempo']
TOP5_GENRES = df['genre'].value_counts().head(5).index.tolist()

print(f" Loaded cleaned dataset: {df.shape}")
print(f"   Genres: {df['genre'].nunique()}  |  Years: {df['release_year'].min()}–{df['release_year'].max()}")


---
## Section 1 — Distribution Analysis

### Chart 1: Popularity Score Distribution


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

sns.histplot(df['popularity'], bins=50, kde=True, color=SPOTIFY_GREEN,
             line_kws={'linewidth': 2.5}, ax=ax)
ax.axvline(df['popularity'].mean(),   color='navy',   linestyle='--', linewidth=1.8,
           label=f"Mean: {df['popularity'].mean():.1f}")
ax.axvline(df['popularity'].median(), color='crimson', linestyle='-', linewidth=1.8,
           label=f"Median: {df['popularity'].median():.1f}")

ax.set_title('Distribution of Track Popularity Scores', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Popularity Score (0–100)', fontsize=12)
ax.set_ylabel('Number of Tracks', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()


**Interpretation:** Popularity is roughly normally distributed around a mean of ~48, meaning most tracks are "average" and truly viral hits are rare. The slight right-skew towards the 60–80 range suggests that well-produced tracks consistently outperform the median.

### Chart 2: Audio Feature Box Plots


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 5))
features_0_1 = ['danceability', 'energy', 'instrumentalness']
features_other = ['loudness', 'tempo']

for i, feat in enumerate(AUDIO_FEATURES):
    ax = axes[i]
    sns.boxplot(y=df[feat], ax=ax, color=PALETTE[i], width=0.5, fliersize=2)
    ax.set_title(feat.capitalize(), fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(feat, fontsize=10)

fig.suptitle('Distribution of Audio Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**Interpretation:** Danceability and energy are evenly spread across their 0–1 range, suggesting a diverse catalogue. Instrumentalness is heavily skewed towards zero — most tracks in this dataset are vocal. Loudness clusters around –10 dB, consistent with modern mastering standards.

### Chart 3: Track Duration Distribution


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

sns.histplot(df['duration_min'], bins=60, kde=True, color=PALETTE[2],
             line_kws={'linewidth': 2}, ax=ax)
ax.axvline(df['duration_min'].mean(),   color='navy',   linestyle='--', linewidth=1.8,
           label=f"Mean: {df['duration_min'].mean():.2f} min")
ax.axvline(df['duration_min'].median(), color='crimson', linestyle='-', linewidth=1.8,
           label=f"Median: {df['duration_min'].median():.2f} min")

ax.set_title('Track Duration Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Duration (minutes)', fontsize=12)
ax.set_ylabel('Number of Tracks', fontsize=12)
ax.set_xlim(0, 12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()


**Interpretation:** Most tracks cluster in the 3–5 minute sweet spot, which aligns with traditional radio-friendly formats. The right tail shows classical and jazz compositions that extend well beyond 6 minutes.

---
## Section 2 — Correlation Analysis

### Chart 4: Correlation Heatmap


In [ ]:
num_cols = ['popularity', 'danceability', 'energy', 'loudness',
            'instrumentalness', 'tempo', 'duration_min', 'stream_millions']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, square=True, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 10}, ax=ax)
ax.set_title('Correlation Matrix — Numerical Features', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


**Interpretation:** Energy and loudness show the strongest positive correlation, which makes physical sense — louder mixes feel more energetic. Instrumentalness is negatively correlated with popularity, confirming that vocal tracks dominate the charts. Popularity has only weak correlations with individual audio features, suggesting hit-making is multifactorial.

### Chart 5: Danceability vs Energy (coloured by Popularity Tier)


In [ ]:
sample = df.sample(5000, random_state=42)
tier_colors = {'Low': '#e74c3c', 'Medium': '#f39c12', 'High': '#1DB954'}

fig, ax = plt.subplots(figsize=(11, 7))
for tier, grp in sample.groupby('popularity_tier', observed=True):
    ax.scatter(grp['danceability'], grp['energy'],
               c=tier_colors[tier], label=tier, alpha=0.45, s=18, edgecolors='none')

ax.set_title('Danceability vs Energy — coloured by Popularity Tier', fontsize=14, fontweight='bold')
ax.set_xlabel('Danceability (0–1)', fontsize=12)
ax.set_ylabel('Energy (0–1)', fontsize=12)
ax.legend(title='Popularity Tier', fontsize=11)
plt.tight_layout()
plt.show()


**Interpretation:** High-popularity tracks (green) are spread across the danceability-energy space but show a slight concentration in the high danceability + medium-high energy quadrant (top-right), suggesting producers aiming for commercial success target this zone.

### Chart 6: Tempo vs Energy by Genre (top 5)


In [ ]:
top5_df = df[df['genre'].isin(TOP5_GENRES)].sample(3000, random_state=42)
genre_palette = dict(zip(TOP5_GENRES, PALETTE[:5]))

fig, ax = plt.subplots(figsize=(11, 7))
for genre, grp in top5_df.groupby('genre'):
    ax.scatter(grp['tempo'], grp['energy'],
               c=[genre_palette[genre]], label=genre, alpha=0.45, s=18, edgecolors='none')

ax.set_title('Tempo vs Energy — Top 5 Genres', fontsize=14, fontweight='bold')
ax.set_xlabel('Tempo (BPM)', fontsize=12)
ax.set_ylabel('Energy (0–1)', fontsize=12)
ax.legend(title='Genre', fontsize=10)
plt.tight_layout()
plt.show()


**Interpretation:** Genres form distinct clusters in tempo-energy space. Metal occupies the high-energy, high-tempo corner while Jazz spreads across lower energy and varied tempos. This confirms that audio features carry meaningful genre signal — useful for future classification tasks.

---
## Section 3 — Categorical Analysis

### Chart 7: Top 15 Artists by Track Count


In [ ]:
top_artists = df['artist_name'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_artists.index[::-1], top_artists.values[::-1],
               color=SPOTIFY_GREEN, alpha=0.85)
ax.bar_label(bars, padding=4, fontsize=10)
ax.set_title('Top 15 Artists by Number of Tracks', fontsize=14, fontweight='bold')
ax.set_xlabel('Track Count', fontsize=12)
ax.set_ylabel('')
ax.set_xlim(0, top_artists.max() * 1.15)
plt.tight_layout()
plt.show()


**Interpretation:** The most prolific artists have a large catalogue in this dataset. High track count doesn't always equal high popularity — we'll explore artist consistency vs. one-hit wonders in Notebook 04.

### Chart 8: Top 10 Genres by Average Popularity


In [ ]:
genre_pop = (df.groupby('genre')['popularity']
               .mean()
               .sort_values(ascending=True)
               .tail(10))

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(genre_pop.index, genre_pop.values,
               color=sns.color_palette('Set2', len(genre_pop)), alpha=0.9)
ax.bar_label(bars, fmt='%.1f', padding=4, fontsize=10)
ax.set_title('Top 10 Genres by Average Popularity', fontsize=14, fontweight='bold')
ax.set_xlabel('Average Popularity Score', fontsize=12)
ax.set_xlim(0, genre_pop.max() * 1.15)
plt.tight_layout()
plt.show()


**Interpretation:** Genre popularity rankings reveal listener preferences at scale. The differences between genres are relatively small, suggesting that audio quality and production matter more than genre alone.

### Chart 9: Popularity Tier Distribution (Donut Chart)


In [ ]:
tier_counts = df['popularity_tier'].value_counts().reindex(['Low','Medium','High'])
colors = ['#e74c3c', '#f39c12', '#1DB954']

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    tier_counts, labels=tier_counts.index, colors=colors,
    autopct='%1.1f%%', pctdistance=0.75,
    wedgeprops=dict(width=0.5, edgecolor='white', linewidth=2),
    startangle=90, textprops={'fontsize': 13}
)
for at in autotexts:
    at.set_fontsize(12)
    at.set_fontweight('bold')

ax.set_title('Track Distribution by Popularity Tier', fontsize=14, fontweight='bold', pad=20)
centre = plt.Circle((0, 0), 0.25, color='white')
ax.add_patch(centre)
plt.tight_layout()
plt.show()


**Interpretation:** The three tiers are distributed roughly evenly by design of our binning (0-33/34-66/67-100), but the exact split reveals whether the popularity distribution is truly symmetric or skewed.

### Chart 10: Average Audio Features by Popularity Tier


In [ ]:
features_norm = ['danceability', 'energy', 'instrumentalness']
tier_means = (df.groupby('popularity_tier', observed=True)[features_norm]
                .mean()
                .reindex(['Low','Medium','High']))

x = np.arange(len(features_norm))
width = 0.25
tier_colors_list = ['#e74c3c', '#f39c12', '#1DB954']

fig, ax = plt.subplots(figsize=(11, 6))
for i, (tier, row) in enumerate(tier_means.iterrows()):
    bars = ax.bar(x + i*width, row.values, width, label=tier,
                  color=tier_colors_list[i], alpha=0.85)
    ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=9)

ax.set_title('Average Audio Features by Popularity Tier', fontsize=14, fontweight='bold')
ax.set_xlabel('Audio Feature', fontsize=12)
ax.set_ylabel('Average Value (0–1 scale)', fontsize=12)
ax.set_xticks(x + width)
ax.set_xticklabels([f.capitalize() for f in features_norm], fontsize=11)
ax.legend(title='Popularity Tier', fontsize=10)
ax.set_ylim(0, 0.75)
plt.tight_layout()
plt.show()


**Interpretation:** High-popularity tracks have notably higher danceability and lower instrumentalness compared to low-popularity tracks. This confirms the commercial formula: danceable, vocal-led tracks outperform instrumental compositions.

---
## Section 4 — Time Series & Trends

### Chart 11: Average Popularity by Release Year


In [ ]:
yearly = (df[df['release_year'].between(2015, 2025)]
           .groupby('release_year')['popularity']
           .agg(['mean', 'std'])
           .reset_index())

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(yearly['release_year'], yearly['mean'],
        marker='o', linewidth=2.5, color=SPOTIFY_GREEN, markersize=8, label='Avg Popularity')
ax.fill_between(yearly['release_year'],
                yearly['mean'] - yearly['std'],
                yearly['mean'] + yearly['std'],
                alpha=0.15, color=SPOTIFY_GREEN, label='±1 Std Dev')

ax.set_title('Average Track Popularity by Release Year (2015–2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Release Year', fontsize=12)
ax.set_ylabel('Average Popularity Score', fontsize=12)
ax.set_xticks(range(2015, 2026))
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()


**Interpretation:** Popularity trends over time reveal whether newer releases are outperforming older catalogues, or whether classic tracks maintain their listener appeal. Rising trends may indicate algorithm-driven rediscovery or shifting listener tastes.

### Chart 12: Genre Track Volume Over Years (Stacked Area)


In [ ]:
genre_year = (df[df['release_year'].between(2015, 2025) & df['genre'].isin(TOP5_GENRES)]
              .groupby(['release_year', 'genre'])
              .size()
              .unstack(fill_value=0))

fig, ax = plt.subplots(figsize=(13, 6))
genre_year.plot.area(ax=ax, colormap='Set2', alpha=0.8, linewidth=0)
ax.set_title('Track Volume by Genre Over Years (Top 5 Genres)', fontsize=14, fontweight='bold')
ax.set_xlabel('Release Year', fontsize=12)
ax.set_ylabel('Number of Tracks', fontsize=12)
ax.legend(title='Genre', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=10)
ax.set_xticks(range(2015, 2026))
plt.tight_layout()
plt.show()


**Interpretation:** The stacked area chart shows how the genre mix has evolved year by year. Rising areas indicate growing production volume in that genre, while shrinking areas may reflect genre consolidation or changing label priorities.

### Chart 13: Danceability Heatmap by Year & Month


In [ ]:
heat_data = (df[df['release_year'].between(2015, 2025)]
             .groupby(['release_year', 'release_month'])['danceability']
             .mean()
             .unstack())

month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']
heat_data.columns = [month_labels[int(m)-1] for m in heat_data.columns]

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(heat_data, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.4, linecolor='white',
            cbar_kws={'label': 'Avg Danceability'}, ax=ax,
            annot_kws={'size': 9})
ax.set_title('Average Danceability by Release Year & Month', fontsize=14, fontweight='bold')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Year', fontsize=12)
plt.tight_layout()
plt.show()


**Interpretation:** This heatmap reveals seasonal and annual danceability trends. Summer months (Jun–Aug) may show higher danceability as labels target festival season. Year-over-year changes reflect evolving production trends in mainstream music.

---
## Section 5 — Advanced Charts

### Chart 14: Pair Plot — Top 4 Audio Features by Popularity Tier


In [ ]:
pair_sample = df.sample(2000, random_state=42)[
    ['danceability', 'energy', 'instrumentalness', 'tempo', 'popularity_tier']
]
tier_pal = {'Low': '#e74c3c', 'Medium': '#f39c12', 'High': '#1DB954'}

g = sns.pairplot(pair_sample, hue='popularity_tier',
                 palette=tier_pal, plot_kws={'alpha': 0.35, 's': 15},
                 diag_kind='kde', corner=True)
g.figure.suptitle('Pair Plot — Audio Features by Popularity Tier', y=1.01,
                   fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


**Interpretation:** The pair plot confirms that no single audio feature perfectly separates popularity tiers — the colour distributions overlap substantially. However, the danceability KDE diagonal shows a slight rightward shift for High popularity, reinforcing its importance as a predictor.

### Chart 15: Popularity Distribution by Top 5 Genres (Violin Plot)


In [ ]:
violin_df = df[df['genre'].isin(TOP5_GENRES)]

fig, ax = plt.subplots(figsize=(13, 7))
sns.violinplot(data=violin_df, x='genre', y='popularity',
               order=TOP5_GENRES, palette='Set2',
               inner='box', linewidth=1.2, ax=ax)
ax.set_title('Popularity Distribution by Top 5 Genres', fontsize=14, fontweight='bold')
ax.set_xlabel('Genre', fontsize=12)
ax.set_ylabel('Popularity Score', fontsize=12)
ax.set_xticklabels(TOP5_GENRES, fontsize=11)
plt.tight_layout()
plt.show()

print("\n EDA complete — 15 visualisations created across 5 sections.")
print("   Next: Notebook 04 — Business Insights")
